# 08 错误分析、从零卷积与坏实验

## 目标
会看误分类图册；能让 NumPy 卷积对齐 PyTorch；能诊断五种经典事故。

## 会错的直觉
「准确率已经 85%，课就结束了。」——剩下的 15% 才告诉你下一步改数据还是改模型。

In [ ]:
import numpy as np, torch, torch.nn.functional as F
from mnist_lab.numpy_conv import conv2d_numpy
img = np.random.randn(1, 1, 8, 8).astype(np.float32)
ker = np.random.randn(2, 1, 3, 3).astype(np.float32)
diff = np.max(np.abs(conv2d_numpy(img, ker, padding=1) - F.conv2d(torch.tensor(img), torch.tensor(ker), padding=1).numpy()))
print('max |numpy-torch|', diff)
assert diff < 1e-4

In [ ]:
from mnist_lab.tiny_data import make_tiny_loaders, DEFAULT_CHECKPOINT
from mnist_lab.models import build_model
from mnist_lab.evaluate import evaluate
from mnist_lab.visualize import save_error_gallery
import torch
model = build_model('cnn')
model.load_state_dict(torch.load(DEFAULT_CHECKPOINT, map_location='cpu'))
_, _, te = make_tiny_loaders(batch_size=32)
ev = evaluate(model, te)
xs, ys = next(iter(te))
with torch.no_grad():
    pred = torch.argmax(model(xs), 1).numpy()
print('acc', round(ev.accuracy, 3), 'saved', save_error_gallery(xs, ys.numpy(), pred, 'outputs/nb08_err.png'))

In [ ]:
from mnist_lab.bugs import list_cases, run_bug_case, grade_diagnosis
print([c.id for c in list_cases()])
p = run_bug_case('exploding_lr', 'runs/nb08_lr')
print(p['highlight'])
assert grade_diagnosis('exploding_lr', 'B')

## 思考题
1. 4 和 9 经常对调时，增强或加数据，哪件事更先做？
2. 白底照片预测全错，你先检查模型还是先看预处理？
3. 把测试集准确率写进训练循环当「验证」，报告还能信吗？